# Lab 1 · Meeting the Model: Baseline Inference

**~30 minutes.** Nothing is trained in this notebook. We are establishing
the "before" — and playing with every inference parameter from the deck.

Two things happen here:

1. We ask the base model twelve questions about a cluster called **AURA**
   and save its answers. AURA doesn't exist, so it cannot possibly know —
   watch it answer confidently anyway.
2. We work through temperature, top-k, top-p, penalties and stop
   sequences hands-on.

> ↳ Slides: *LLM Parameters* · *entire Inference Parameters section*

In [ ]:
# --- Install the fine-tuning stack --------------------------------------
# Kaggle ships a matched torch/CUDA pair. --no-deps stops pip replacing torch
# with an incompatible build, which shows up later as baffling CUDA errors.
#
# Takes 2-4 minutes. "dependency conflict" warnings here are expected and fine.
# Output is deliberately NOT suppressed: if this step fails, you need to see it.
!pip install -q --no-deps unsloth unsloth_zoo
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets huggingface_hub sentencepiece protobuf

print("\ninstall finished - verifying imports...")
import importlib
missing = [m for m in ("unsloth", "trl", "peft", "bitsandbytes", "datasets")
           if importlib.util.find_spec(m) is None]
print("MISSING: " + ", ".join(missing) if missing else "all packages importable")

In [ ]:
# --- Locate the workshop repo -------------------------------------------
# Tries, in order: already present -> attached Kaggle Dataset -> git clone.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/YOUR-USERNAME/LLM-lab.git"   # TODO: your repo

def find_repo() -> Path:
    for candidate in [Path("/kaggle/working/LLM-lab"), Path.cwd(), Path.cwd().parent]:
        if (candidate / "common" / "config.py").exists():
            return candidate
    for d in Path("/kaggle/input").glob("*"):          # attached as a Dataset
        if (d / "common" / "config.py").exists():
            return d
    print("Repo not found locally, cloning...")        # last resort
    subprocess.run(["git", "clone", "-q", REPO_URL, "/kaggle/working/LLM-lab"], check=True)
    return Path("/kaggle/working/LLM-lab")

REPO = find_repo()
sys.path[:0] = [str(REPO / "common"), str(REPO / "dataset")]
print(f"repo: {REPO}")

import config
print(config.summary())

In [ ]:
# --- Hugging Face authentication ----------------------------------------
# Reads the Kaggle Secret named HF_TOKEN. Never paste a token into a cell:
# it is saved with the notebook and shared notebooks leak tokens constantly.
import os

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("Hugging Face: authenticated")
except Exception as e:
    print(f"Hugging Face: NOT authenticated ({type(e).__name__})")
    print("  Fix: right panel -> Add-ons -> Secrets -> add HF_TOKEN, tick the box.")
    print("  Or set USE_UNGATED_MODEL = True below to skip the gated model entirely.")

## 2.1 Load the model in 4-bit

`load_in_4bit=True` is the Q in QLoRA. Each weight, normally 16 bits, is
stored in 4 — a 4× reduction that is what makes a 3B model fit
comfortably on a free T4.

`dtype=None` lets Unsloth pick fp16 or bf16 from your GPU. On a T4 it
picks fp16, as Lab 0 predicted.

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = config.MODEL_NAME,
    max_seq_length = config.MAX_SEQ_LENGTH,
    dtype          = None,               # None = auto-detect from the GPU
    load_in_4bit   = config.LOAD_IN_4BIT,
)
print(f"\nloaded {config.MODEL_NAME}")
print(f"dtype in use: {model.dtype}")

## 2.2 Measure the VRAM — check the arithmetic from the slides

The theory said: **model size ≈ parameter count × bytes per parameter.**

Llama-3.2-3B has 3.21 billion parameters. At 4 bits (0.5 bytes) that's
roughly 1.6 GB, plus overhead for the parts that stay in higher precision
(embeddings, layer norms). Let's see what actually landed on the card.

> ↳ Slide: *Model Size based on Parameter Count and Precision*

In [ ]:
import torch

def vram(label=""):
    used = torch.cuda.memory_allocated() / 2**30
    peak = torch.cuda.max_memory_allocated() / 2**30
    print(f"  {label:<28} {used:5.2f} GB in use   (peak {peak:5.2f} GB)")
    return used

n_params = sum(p.numel() for p in model.parameters())

print(f"  parameters                  {n_params/1e9:.2f} B")
print(f"  if stored at fp16 (2 B)     {n_params*2/2**30:5.2f} GB")
print(f"  if stored at 4-bit (0.5 B)  {n_params*0.5/2**30:5.2f} GB")
print()
vram("actually on the GPU")
print("\n  The real number sits above the 4-bit estimate because")
print("  embeddings and norms stay in higher precision.")

## 2.3 Where did the parameters go?

The slides listed what a "parameter" actually is: weights, biases, token
embeddings. Let's look at the real tensors and group them.

Notice how much of the model is the **embedding table** — vocabulary size
× hidden dimension is a surprisingly large slice for a small model.

> ↳ Slide: *Parameters: weights, biases, token embedding, KV cache*

In [ ]:
from collections import defaultdict

groups = defaultdict(int)
for name, p in model.named_parameters():
    if "embed" in name:            key = "token embeddings"
    elif "lm_head" in name:        key = "output head"
    elif "norm" in name:           key = "layer norms"
    elif any(k in name for k in ("q_proj","k_proj","v_proj","o_proj")):
                                   key = "attention (Q,K,V,O)"
    elif any(k in name for k in ("gate_proj","up_proj","down_proj")):
                                   key = "MLP (gate,up,down)"
    else:                          key = "other"
    groups[key] += p.numel()

total = sum(groups.values())
print(f"  {'component':<22} {'params':>12}   share")
print("  " + "-"*48)
for k, v in sorted(groups.items(), key=lambda x: -x[1]):
    print(f"  {k:<22} {v/1e6:>9.1f} M   {v/total:>6.1%}")
print("  " + "-"*48)
print(f"  {'total':<22} {total/1e9:>9.2f} B")

## 2.4 Make the chat template visible

This is the single most under-appreciated detail in fine-tuning.

An instruct model was not trained on raw text — it was trained on text
wrapped in **special tokens** marking who is speaking. `<|start_header_id|>`,
`<|eot_id|>` and friends aren't decoration; the model learned that an
answer follows one specific token sequence.

Get this wrong and nothing crashes. The model just gets quietly worse, and
you spend a day blaming your learning rate.

> ↳ Slide: *Instruct vs Base*

In [ ]:
messages = [
    {"role": "system",  "content": "You are a helpful assistant."},
    {"role": "user",    "content": "What is 2+2?"},
]

templated = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

print("What the model actually receives:")
print("=" * 66)
print(templated)
print("=" * 66)
print("\nAs token IDs:")
ids = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
for i in ids[:14]:
    print(f"  {i:>6}  {tokenizer.decode([i])!r}")
print(f"  ... {len(ids)} tokens total")

## 2.5 First generation

A small helper we'll reuse all notebook. Note `FastLanguageModel.for_inference` —
it enables Unsloth's ~2× faster inference path.

In [ ]:
FastLanguageModel.for_inference(model)

def ask(question, system=config.SYSTEM_PROMPT, **kw):
    """Generate an answer. Any sampling kwarg can be overridden."""
    msgs = ([{"role": "system", "content": system}] if system else []) + \
           [{"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    params = dict(
        max_new_tokens = config.MAX_NEW_TOKENS,
        temperature    = config.TEMPERATURE,
        top_p          = config.TOP_P,
        top_k          = config.TOP_K,
        do_sample      = True,
        pad_token_id   = tokenizer.eos_token_id,
    )
    params.update(kw)
    if params.get("temperature") == 0:            # greedy decoding
        params.update(do_sample=False, temperature=None, top_p=None, top_k=None)

    out = model.generate(input_ids=inputs, **params)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

print(ask("Explain what a GPU is in two sentences."))

## 2.6 The baseline evaluation — the "before" half of the demo

Twelve questions about the **AURA cluster**. AURA is fictional: its
partition names, module versions, quotas and wrapper commands appear in no
pretraining corpus anywhere.

A well-behaved model would say "I don't know about AURA." Watch what
actually happens.

**Read a few of these answers properly.** They are fluent, plausible,
formatted like real documentation, and completely invented. That is the
failure mode fine-tuning fixes — and it's why "the model sounds confident"
tells you nothing.

*This takes 2–3 minutes.*

In [ ]:
from eval_prompts import EVAL_PROMPTS
import compare

baseline = {}
for i, p in enumerate(EVAL_PROMPTS, 1):
    print(f"[{i:>2}/{len(EVAL_PROMPTS)}] {p['question'][:58]}...")
    baseline[p["id"]] = ask(p["question"], temperature=0.0)   # greedy = reproducible

compare.save(baseline, config.BASELINE_ANSWERS)

In [ ]:
# Look at three of them in full.
for p in EVAL_PROMPTS[:3]:
    print("=" * 72)
    print("Q:", p["question"])
    print("-" * 72)
    print(baseline[p["id"]][:500])
    print()

In [ ]:
# Score it. Expect a very low number - that is the point.
scores = compare.score_all(baseline)
print(f"  baseline: {sum(scores.values())}/{len(scores)} correct\n")
for p in EVAL_PROMPTS:
    print(f"    {'PASS' if scores[p['id']] else 'FAIL'}  {p['question'][:56]}")

---
# 2.7 Inference Parameter Playground

Everything in the *Inference Parameters* section of the deck, hands-on.

The model is now **fixed** — we are not changing a single weight. Every
difference you see from here on comes from how we *sample* from the same
probability distribution.

### Temperature

At each step the model produces a score (logit) for every token in the
vocabulary. Temperature divides those logits before they become
probabilities:

- **T → 0** — the gap between best and second-best widens; always picks the top token. Deterministic.
- **T = 1** — the model's own distribution, untouched.
- **T > 1** — flattens the distribution; unlikely tokens get a real chance. Creative, then incoherent.

Same prompt, four temperatures.

In [ ]:
prompt = "Describe a thunderstorm in one sentence."

for t in (0.0, 0.3, 0.7, 1.5):
    print(f"--- temperature = {t} ---")
    print(ask(prompt, temperature=t, max_new_tokens=60).strip(), "\n")

### Seeing top-k and top-p rather than being told about them

Both truncate the candidate list before sampling:

- **top-k = 40** — keep the 40 most likely tokens, discard the rest.
- **top-p = 0.9** — keep the smallest set of tokens whose probabilities sum to 0.9. The list size *adapts*: tiny when the model is confident, large when it isn't.

Let's print the actual distribution for one prediction and mark where each
cutoff falls.

In [ ]:
import torch.nn.functional as F

msgs = [{"role": "user", "content": "The capital of France is"}]
ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True,
                                    return_tensors="pt").to("cuda")
with torch.no_grad():
    logits = model(ids).logits[0, -1, :]

probs = F.softmax(logits.float(), dim=-1)
top = torch.topk(probs, 15)

cumulative = 0.0
print(f"  {'rank':>4} {'token':<16} {'prob':>8} {'cumulative':>11}")
print("  " + "-" * 46)
for rank, (p_, idx) in enumerate(zip(top.values, top.indices), 1):
    cumulative += p_.item()
    marks = []
    if rank == 10:                      marks.append("<- top_k=10 cuts here")
    if cumulative >= 0.9 and cumulative - p_.item() < 0.9:
                                        marks.append("<- top_p=0.9 cuts here")
    print(f"  {rank:>4} {tokenizer.decode([idx]):<16} "
          f"{p_.item():>8.4f} {cumulative:>11.4f}  {' '.join(marks)}")

print(f"\n  Vocabulary size: {len(probs):,} tokens.")
print(f"  The top 15 hold {top.values.sum().item():.1%} of the probability mass.")
print("  Everything else is noise we throw away before sampling.")

### Repetition: presence vs frequency penalty

Small models fall into loops. Two different fixes, often confused:

- **presence penalty** — a flat penalty once a token has appeared *at all*. Pushes toward new topics.
- **frequency penalty** — scales with *how often* the token appeared. Punishes the tenth repeat far harder than the first.

HuggingFace exposes `repetition_penalty`, which is closest to a presence
penalty. We'll provoke a loop and then damp it.

In [ ]:
loop_prompt = "List the colour red, over and over, forever:"

for rp in (1.0, 1.15, 1.4):
    label = "no penalty" if rp == 1.0 else f"repetition_penalty={rp}"
    print(f"--- {label} ---")
    print(ask(loop_prompt, repetition_penalty=rp,
              max_new_tokens=70, temperature=0.8).strip()[:280], "\n")

### max_new_tokens, stop sequences, and the context window

Three separate limits people routinely confuse:

| | what it limits |
|---|---|
| **context window** | prompt + generation combined. A property of the *model* (128k for Llama 3.2). |
| **max_new_tokens** | how much is generated *this call*. Your choice. |
| **stop / EOS** | content-based early exit, regardless of the other two. |

Truncation mid-sentence is nearly always `max_new_tokens`, not the context
window.

In [ ]:
print(f"  model context window : {model.config.max_position_embeddings:,} tokens")
print(f"  configured for today : {config.MAX_SEQ_LENGTH:,} tokens")
print(f"  EOS token            : {tokenizer.eos_token!r} (id {tokenizer.eos_token_id})\n")

q = "Count from 1 to 30, one number per line."
for n in (20, 120):
    print(f"--- max_new_tokens = {n} ---")
    print(ask(q, max_new_tokens=n, temperature=0.0).strip()[:220], "\n")

print("The first one stops mid-count: it hit the token budget, not the")
print("end of the answer. Nothing errors - you just get a truncated string.")

## 2.8 The KV cache, observed

Generating token 500 requires attending to the previous 499. Without a
cache you'd recompute all their key and value vectors every single step —
quadratic work for a linear output.

The KV cache stores them. It costs memory that grows with sequence length,
which is why the slides listed it alongside weights as a real consumer of
VRAM — and why serving engines like vLLM (Lab 7) exist mainly to manage it.

> ↳ Slide: *Parameters / KV cache*

In [ ]:
import time

q = "Write a short paragraph about mountains."
for use_cache in (True, False):
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    _ = ask(q, max_new_tokens=100, temperature=0.0, use_cache=use_cache)
    dt = time.time() - t0
    print(f"  use_cache={str(use_cache):<5}  {dt:5.1f} s   "
          f"peak {torch.cuda.max_memory_allocated()/2**30:.2f} GB")

print("\n  Same output, several times the wall-clock without the cache.")
print("  It trades memory for not redoing work - the core tradeoff in serving.")

## What we established

- The base model **cannot** answer AURA questions, and does not say so — it invents fluent, well-formatted, wrong answers.
- Sampling parameters change output substantially without touching a single weight.
- `baseline_answers.json` is saved. **Lab 4 needs it** — don't delete it.

Next we build the dataset that fixes this.

---

### Next: `02_dataset_prep.ipynb` — four dataset formats, and the chat template

> **Kaggle tip:** if the session has been idle a while, check the right-hand
> panel still shows the GPU attached before starting the next notebook.